In [1]:
# go to /content first, not inside the repo folder
%cd /content

# remove any old nested clone if needed
!rm -rf Visual-Saliency

# clone once, into /content/Visual-Saliency
!git clone https://github.com/JesperLybeck/Visual-Saliency.git

%cd /content/Visual-Saliency


/content
Cloning into 'Visual-Saliency'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 21 (delta 7), reused 19 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 6.55 KiB | 6.55 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Visual-Saliency


In [2]:
import kagglehub
from pathlib import Path
import os


print(Path.cwd())

print(Path.cwd())
print(Path.cwd())
print((Path.cwd() / "dataset" / "salicon").exists())

project_dir = Path.cwd().resolve()
dataset_dir = project_dir / "dataset"
download_dir = dataset_dir / "salicon"
path = kagglehub.dataset_download(
        "roshan401/salicon",
        output_dir=str(download_dir),
        force_download=True
    )
print("Downloaded to:", path)
print("Exists:", Path(path).exists())

/content/Visual-Saliency
/content/Visual-Saliency
/content/Visual-Saliency
False
Using Colab cache for faster access to the 'salicon' dataset.
Downloaded to: /kaggle/input/salicon
Exists: True


In [3]:
import torch 


print("CUDA available:", torch.cuda.is_available())
print("MPS available:", getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available())

CUDA available: False
MPS available: False


In [4]:
from resNet import SaliencyResNet
from dataLoader import get_dataloaders
net = SaliencyResNet()


DATASET_ROOT = Path("/kaggle/input/salicon")
train_loader, val_loader, test_loader = get_dataloaders(batch_size=32, dataset_root=DATASET_ROOT)

Getting dataloaders...
Image dir: /kaggle/input/salicon/images/images/train
Image dir exists: True
Heatmap dir: /kaggle/input/salicon/maps/train
Heatmap dir exists: True
Matched 10000 pairs

Image dir: /kaggle/input/salicon/images/images/val
Image dir exists: True
Heatmap dir: /kaggle/input/salicon/maps/val
Heatmap dir exists: True
Matched 5000 pairs

Image dir: /kaggle/input/salicon/images/images/train
Image dir exists: True
Heatmap dir: /kaggle/input/salicon/maps/train
Heatmap dir exists: True
Matched 10000 pairs



In [6]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from typing import Optional, Any, Dict

def _to_device(batch, device):
    x, y = batch
    return x.to(device), y.to(device)

def kl_spatial_loss(preds: torch.Tensor, targets: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    preds: (B,1,H,W) raw output (logits)
    targets: (B,1,H,W) or (B,H,W) ground-truth non-negative maps
    Returns: scalar KL divergence (batchmean)
    """
    B = preds.shape[0]
    pred_flat = preds.view(B, -1)          # (B, HW)
    tgt_flat = targets.view(B, -1).float() # (B, HW)

    # normalize target to sum=1 per sample (probabilities)
    tgt_sum = tgt_flat.sum(dim=1, keepdim=True)
    tgt_prob = tgt_flat / (tgt_sum + eps)

    # handle empty maps (all zeros) by using uniform distribution
    zero_mask = (tgt_sum.squeeze(1) == 0)
    if zero_mask.any():
        tgt_prob[zero_mask] = 1.0 / tgt_prob.shape[1]

    # compute log-softmax over spatial locations for preds
    pred_log = F.log_softmax(pred_flat, dim=1)

    return F.kl_div(pred_log, tgt_prob, reduction="batchmean")

def batch_cc(preds: torch.Tensor, targets: torch.Tensor, eps: float = 1e-8) -> float:
    # returns mean Pearson CC over batch
    if preds.ndim == 4 and preds.shape[1] == 1:
        p = preds.view(preds.shape[0], -1)
    else:
        p = preds.view(preds.shape[0], -1)
    t = targets.view(targets.shape[0], -1).float()

    p_mean = p.mean(dim=1, keepdim=True)
    t_mean = t.mean(dim=1, keepdim=True)
    p_z = p - p_mean
    t_z = t - t_mean

    num = (p_z * t_z).sum(dim=1)
    den = torch.sqrt((p_z ** 2).sum(dim=1) * (t_z ** 2).sum(dim=1) + eps)
    cc = num / den
    return float(cc.mean().item())

def train_one_epoch(model: torch.nn.Module,
                    dataloader: torch.utils.data.DataLoader,
                    optimizer: torch.optim.Optimizer,
                    device: str,
                    clip_grad: Optional[float] = None) -> Dict[str, float]:
    model.train()
    running_loss = 0.0
    running_cc = 0.0
    n = 0
    pbar = tqdm(dataloader, desc="Train", leave=False)
    for batch in pbar:
        x, y = _to_device(batch, device)
        optimizer.zero_grad()
        preds = model(x)  # (B,1,H,W)

        loss = kl_spatial_loss(preds, y)
        loss.backward()
        if clip_grad is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()

        bs = x.shape[0]
        running_loss += loss.item() * bs
        running_cc += batch_cc(preds.detach(), y.detach()) * bs
        n += bs
        pbar.set_postfix(loss=loss.item())

    return {"loss": running_loss / n, "cc": running_cc / n}

def validate(model: torch.nn.Module,
             dataloader: torch.utils.data.DataLoader,
             device: str) -> Dict[str, float]:
    model.eval()
    running_loss = 0.0
    running_cc = 0.0
    n = 0
    pbar = tqdm(dataloader, desc="Valid", leave=False)
    with torch.no_grad():
        for batch in pbar:
            x, y = _to_device(batch, device)
            preds = model(x)
            loss = kl_spatial_loss(preds, y)

            bs = x.shape[0]
            running_loss += loss.item() * bs
            running_cc += batch_cc(preds, y) * bs
            n += bs
            pbar.set_postfix(loss=loss.item())

    return {"loss": running_loss / n, "cc": running_cc / n}

def fit(model: torch.nn.Module,
        train_loader: torch.utils.data.DataLoader,
        valid_loader: torch.utils.data.DataLoader,
        optimizer: torch.optim.Optimizer,
        device: str,
        epochs: int = 10,
        save_path: str = "best_saliency_kl.pt",
        clip_grad: Optional[float] = None,
        scheduler: Optional[Any] = None) -> Dict[str, list]:
    history = {"train_loss": [], "train_cc": [], "val_loss": [], "val_cc": []}
    best_val = float("inf")
    model.to(device)

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        train_metrics = train_one_epoch(model, train_loader, optimizer, device, clip_grad)
        val_metrics = validate(model, valid_loader, device)

        if scheduler is not None:
            try:
                scheduler.step(val_metrics["loss"])
            except Exception:
                try:
                    scheduler.step()
                except Exception:
                    pass

        history["train_loss"].append(train_metrics["loss"])
        history["train_cc"].append(train_metrics["cc"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_cc"].append(val_metrics["cc"])

        print(f" Train loss: {train_metrics['loss']:.4f}, CC: {train_metrics['cc']:.4f}")
        print(f" Val   loss: {val_metrics['loss']:.4f}, CC: {val_metrics['cc']:.4f}")

        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            torch.save(model.state_dict(), save_path)
            print(f" Saved best model to {save_path}")

    return history

# Example usage:

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
model = SaliencyResNet()  # from resNet.py
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
history = fit(model, train_loader, val_loader, opt, device=device, epochs=20, save_path="best.pt")

Epoch 1/20


KeyboardInterrupt: 